# **NASA 터보팬 엔진 시계열 데이터 분석 및 LSTM 모델 학습**

* **NASA 터보팬 엔진 성능 저하 시뮬레이션 데이터셋**은 예측 유지보수 분야에서 널리 사용되는 벤치마크 데이터입니다.
* 이 데이터를 활용하여 **시계열 데이터의 전처리, 특성공학, LSTM 모델 학습 과정**을 실습합니다.


## **데이터셋 개요**



  * **구성**
    * 데이터셋은 여러 하위 데이터셋(FD001 \~ FD004)으로 나뉘어 있으며,
    * 각각 다른 운영 조건과 고장 모드를 시뮬레이션합니다.
    * 본 가이드에서는 가장 기본적인 **FD001**을 기준으로 설명합니다.
  * **파일:**
      * `train_FD001.txt`: 여러 엔진의 시작부터 고장 시점까지 전체 시계열 데이터.
      * `test_FD001.txt`: 고장 발생 전 임의의 시점에서 데이터가 중단된 엔진들의 시계열 데이터.
      * `RUL_FD001.txt`: `test_FD001.txt`에 포함된 각 엔진의 실제 잔여 수명.



## **데이터 구조**

* 각 파일은 공백으로 구분된 텍스트 파일이며, 각 행은 한 번의 운영 사이클을 의미합니다.
* 주요 컬럼은 다음과 같습니다.

    1.  **Unit number**: 엔진 ID
    2.  **Time in cycles**: 운영 사이클
    3.  **Operational settings (1-3)**: 3개의 운영 설정 값
    4.  **Sensor measurements (1-21)**: 21개의 센서 측정값

# **0. 실습 데이터 준비**

* 실습 데이터는 노트북 시작 부분에서 한 번만 불러옵니다.

* 원본 데이터는 보존하고, 실습 목적에 따라 복사하여 사용합니다.

* 앞에서 실습한 내용을 다시 실행할 때는 해당 실습의 초기화 셀부터 실행합니다.

* 런타임을 다시 시작했다면 0단계 공통 실습 준비부터 순서대로 실행해야 합니다.

## **0-1 데이터 로딩**

In [ ]:
#----- ① 라이브러리 불러오기 및 Google Drive 연결 -----

# 공통으로 사용하는 라이브러리 불러오기
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 구글 드라이브 연결하기
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#----- ② NASA 데이터 로딩 -----

# 컬럼 이름 지정하기
columns_name = (
    ['unit_number', 'cycles',
     'op_setting_1', 'op_setting_2', 'op_setting_3']
    + [f'sensor_{i}' for i in range(1, 22)]
)

# 데이터 파일 경로
data_path = '/content/drive/MyDrive/_lec_data/_PdM/NASA/'   ###### 본인에게 맞게 수정 #######

# 학습 및 테스트 데이터 불러오기
df_raw = pd.read_csv(
    data_path + 'train_FD001.txt',
    sep=r'\s+',
    header=None,
    names=columns_name
)

test_raw = pd.read_csv(
    data_path + 'test_FD001.txt',
    sep=r'\s+',
    header=None,
    names=columns_name
)

# 잔여수명 정답 데이터
rul_raw = pd.read_csv(
    data_path + 'RUL_FD001.txt',
    sep=r'\s+',
    header=None
)

print("학습 데이터:", df_raw.shape)
print("테스트 데이터:", test_raw.shape)
print("RUL 정답 데이터:", rul_raw.shape)

display(df_raw.head())

## **0-2 데이터 초기화**

In [ ]:
#----- ③ 데이터 복사 및 초기화 -----

# 시계열 전처리 및 특성공학 실습용 데이터
df_work = df_raw.copy()

print("전처리 실습 데이터 준비 완료")

## **0-3 주요 변수 이름과 용도**


In [ ]:
# * df_raw  	: 원본 학습 데이터 보관
# * test_raw	: 원본 테스트 데이터 보관
# * rul_raw 	: 테스트 데이터의 RUL 정답
# * df_work 	: 결측치·노이즈·특성공학 실습
# * train_df	: LSTM 학습용 데이터
# * test_df 	: LSTM 평가용 데이터

# **1. 데이터셋 이해**

가장 먼저 데이터셋의 구조를 파악하고 실습에 필요한 파일을 준비해야 합니다.


## **1-1 데이터 기본 구조 확인**

NASA C-MAPSS 데이터는 엔진 ID, 운전 주기, 운전 설정 3개, 센서 측정값 21개로 구성돼 있어. FD001 학습 데이터에는 100대의 엔진이 포함돼 있고, 각 엔진의 데이터는 고장 시점까지 기록되어 있어.

In [ ]:
# 데이터 확인하기

display(df_raw.head())
print("데이터 크기:", df_raw.shape)
print("엔진 수:", df_raw["unit_number"].nunique())

# 행 하나가 엔진 한 대의 특정 운전 주기에서 측정한 데이터이다.

## **1-2 엔진마다 운전 기간이 다를까?**

In [ ]:
# 엔진별 마지막 운전 주기
engine_life = df_raw.groupby("unit_number")["cycles"].max()

print(engine_life.describe())

engine_life.hist(bins=20, figsize=(8, 4))
plt.title("Engine Life Distribution")
plt.xlabel("Last Cycle")
plt.ylabel("Number of Engines")
plt.show()

# cycle은 실제 시간(초·분)이 아니라 운전 주기이다.

## **1-3 엔진 한 대의 센서값은 어떻게 변할까?**

In [ ]:
# 1번 엔진 데이터
engine = df_raw[df_raw["unit_number"] == 1]

plt.figure(figsize=(10, 4))

plt.plot(engine["cycles"], engine["sensor_2"], label="Sensor 2")
plt.plot(engine["cycles"], engine["sensor_7"], label="Sensor 7")

plt.xlabel("Cycle")
plt.ylabel("Sensor Value")
plt.title("Engine 1 - Sensor Measurements")
plt.legend()
plt.grid()
plt.show()

# 두 센서의 값 범위가 서로 다르다면 변화가 잘 드러나지 않을 수도 있다.
# 만약 그렇다면 다음 전처리 과정에서 스케일링이 필요해진다.

# **2. 시계열 데이터의 전처리**

## **2-1. 결측치 처리**

In [ ]:
####### 사용할 데이터 가져오기 ##########

# 2. 1번 엔진의 'sensor_2' 데이터만 따로 복사하여 실습 진행
engine_1_sensor_2 = df_work[df_work['unit_number'] == 1][['cycles', 'sensor_2']].copy()

In [ ]:
# 원본 데이터 확인 (시각화)
plt.figure(figsize=(14, 6))

plt.plot(engine_1_sensor_2['cycles'], engine_1_sensor_2['sensor_2'], 'o-', label='Sensor 2 with Missing Values')

plt.title('Sensor 2 Data with Original Values')
plt.xlabel('Cycles')
plt.ylabel('Sensor Value')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# 3. 인위적으로 결측치 만들기 (예: 50~60 사이클 구간의 데이터를 삭제)
engine_1_sensor_2.loc[50:60, 'sensor_2'] = np.nan

# 4. 결측치 확인 (시각화)
plt.figure(figsize=(14, 6))
plt.plot(engine_1_sensor_2['cycles'], engine_1_sensor_2['sensor_2'], 'o-', label='Sensor 2 with Missing Values')
plt.title('Sensor 2 Data with Artificial Missing Values')
plt.xlabel('Cycles')
plt.ylabel('Sensor Value')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# --- 이제 결측치를 채워봅니다 ---

# 5-1. Forward Fill 적용
engine_1_sensor_2['sensor_2_ffill'] = engine_1_sensor_2['sensor_2'].ffill()

# 5-2. 선형 보간 적용
engine_1_sensor_2['sensor_2_interpolated'] = engine_1_sensor_2['sensor_2'].interpolate(method='linear')


In [ ]:
# 6. 결과 비교 시각화
plt.figure(figsize=(14, 8))
# 원본 + 결측치
#plt.plot(engine_1_sensor_2['cycles'], engine_1_sensor_2['sensor_2'], 'o', label='Original with NaN')
# ffill 결과
plt.plot(engine_1_sensor_2['cycles'], engine_1_sensor_2['sensor_2_ffill'], 's-', label='Forward Fill')
# 선형 보간 결과
plt.plot(engine_1_sensor_2['cycles'], engine_1_sensor_2['sensor_2_interpolated'], '^-', label='Linear Interpolation')
# 원본 + 결측치
plt.plot(engine_1_sensor_2['cycles'], engine_1_sensor_2['sensor_2'], 'o', label='Original with NaN')
plt.title('Comparison of Missing Value Imputation Methods')
plt.xlabel('Cycles')
plt.ylabel('Sensor Value')
plt.legend()
plt.grid(True)
plt.show()

## **2-2 노이즈 제거**

In [ ]:
####### 사용할 데이터 가져오기 ##########

# 1. 1번 엔진의 'sensor_7' 데이터 사용 (변동성이 비교적 큰 센서)
engine_1_sensor_7 = df_work[df_work['unit_number'] == 1][['cycles', 'sensor_7']].copy()

In [ ]:
# 2. 다양한 윈도우 크기로 이동 평균 계산
window_size_small = 5
window_size_large = 20

engine_1_sensor_7['rolling_mean_small'] = engine_1_sensor_7['sensor_7'].rolling(window=window_size_small).mean()
engine_1_sensor_7['rolling_mean_large'] = engine_1_sensor_7['sensor_7'].rolling(window=window_size_large).mean()

# 3. 결과 비교 시각화
plt.figure(figsize=(14, 8))
# 원본 데이터
plt.plot(engine_1_sensor_7['cycles'], engine_1_sensor_7['sensor_7'], label='Original Sensor 7', alpha=0.5)
# 작은 윈도우 이동 평균
plt.plot(engine_1_sensor_7['cycles'], engine_1_sensor_7['rolling_mean_small'], label=f'Rolling Mean (window={window_size_small})')
# 큰 윈도우 이동 평균
plt.plot(engine_1_sensor_7['cycles'], engine_1_sensor_7['rolling_mean_large'], label=f'Rolling Mean (window={window_size_large})')

plt.title('Comparison of Smoothing Effects by Window Size')
plt.xlabel('Cycles')
plt.ylabel('Sensor Value')
plt.legend()
plt.grid(True)
plt.show()

# **3. 시계열 데이터의 특성공학**

## **3-1 롤링 통계 : 데이터의 흐름에서 패턴 찾기**

In [ ]:
# 2. 롤링 통계를 계산할 윈도우 크기 설정
window_size = 10

# 3. 이동 평균과 이동 표준편차 특징 추가
# 중요: groupby('unit_number')로 엔진별로 따로 계산해야 함!
df_work['sensor_3_roll_mean'] = df_work.groupby('unit_number')['sensor_3'].transform(
    lambda x: x.rolling(window_size, min_periods=1).mean()
)
df_work['sensor_3_roll_std'] = df_work.groupby('unit_number')['sensor_3'].transform(
    lambda x: x.rolling(window_size, min_periods=1).std()
)

In [ ]:
### 설명용 코드 : 그룹별 롤링 계산 과정 확인 ####

g_df = df_work.groupby('unit_number')
g_df.first()


In [ ]:
g_df.get_group(2)

In [ ]:
g_df['sensor_3'].rolling(window_size, min_periods=1).mean()

In [ ]:
g_df['sensor_3'].rolling(window_size).mean()

In [ ]:
df_work.groupby('unit_number')['sensor_3'].rolling(window_size, min_periods=1).mean()

In [ ]:
# 4. 특정 엔진(예: 1번 엔진)의 데이터 시각화로 결과 확인
engine_1_df = df_work[df_work['unit_number'] == 1]

plt.figure(figsize=(14, 6))
plt.plot(engine_1_df['cycles'], engine_1_df['sensor_3'], label='Original Sensor 3', color='skyblue')
plt.plot(engine_1_df['cycles'], engine_1_df['sensor_3_roll_mean'],
         label=f'Rolling Mean (window={window_size})', color='orange')
plt.legend()
plt.title('Sensor 3 Original vs. Rolling Mean for Engine 1')
plt.xlabel('Cycles')
plt.ylabel('Sensor Value')
plt.show()

In [ ]:
# 4. 특정 엔진(예: 1번 엔진)의 데이터 시각화로 결과 확인
engine_1_df = df_work[df_work['unit_number'] == 1]

plt.figure(figsize=(14, 6))
plt.plot(engine_1_df['cycles'], engine_1_df['sensor_3_roll_std'],
         label=f'Rolling Std (window={window_size})', color='yellowgreen')
plt.legend()
plt.title('Rolling Std for Engine 1')
plt.xlabel('Cycles')
plt.ylabel('Rolling Std')
plt.show()

In [ ]:
# 4. 특정 엔진(예: 1번 엔진)의 데이터 시각화로 결과 확인
engine_1_df = df_work[df_work['unit_number'] == 1]

plt.rcParams['figure.figsize'] = (14, 6)

fig, ax1 = plt.subplots()
ax1.plot(engine_1_df['cycles'], engine_1_df['sensor_3'], label='Original Sensor 3', color='skyblue')
ax1.plot(engine_1_df['cycles'], engine_1_df['sensor_3_roll_mean'],
         label=f'Rolling Mean (window={window_size})', color='orange')
ax1.set_xlabel('Cycles')
ax1.set_ylabel('Sensor Value')

ax2 = ax1.twinx()
ax2.plot(engine_1_df['cycles'], engine_1_df['sensor_3_roll_std'],
         label=f'Rolling Std (window={window_size})', color='yellowgreen') #linestyle='--'
ax2.set_ylabel('Rolling Std')

plt.show()

## **3-2 지연 특징과 누적 특징 : 과거와 현재를 연결하기**

In [ ]:
# 1. 지연 특징 추가 (1 사이클 전 sensor_4 값)
# 중요: 역시 groupby()를 사용하여 엔진별로 계산해야 데이터가 섞이지 않음
df_work['sensor_4_lag_1'] = df_work.groupby('unit_number')['sensor_4'].transform(lambda x: x.shift(1))

# 2. 누적 합 특징 추가
df_work['sensor_7_cumsum'] = df_work.groupby('unit_number')['sensor_7'].transform(lambda x: x.cumsum())

In [ ]:
# 3-1. 결과 확인 : sensor_7
df_work[['unit_number', 'cycles', 'sensor_7', 'sensor_7_cumsum']].head()

In [ ]:
# 3-2. 결과 확인 : sensor_4
# shift()를 사용하면 첫 번째 행에 결측치(NaN)가 생기므로 처리 필요
print("Lag feature 생성 후 결측치 확인:")

# 결측치가 있음을 확인하기
# 지연 특징의 첫 번째 값은 이전 시점이 없으므로 NaN 발생
df_work[['unit_number', 'cycles', 'sensor_4', 'sensor_4_lag_1']].head()

In [ ]:
# 결측치 처리
# 여기서는 실습 예시로 0을 사용해 결측치를 채움
df_work["sensor_4_lag_1"] = (
    df_work["sensor_4_lag_1"].fillna(0)
)

In [ ]:
# 결과 확인
df_work[['unit_number', 'cycles', 'sensor_4', 'sensor_4_lag_1']].head()

# **4. LSTM을 이용한 터보팬 엔진 잔여수명 예측**

이번에는 NASA 터보팬 엔진 데이터를 LSTM 모델에 입력할 수 있는 형태로 변환하고, 엔진의 잔여수명(RUL)을 예측합니다.

**실습 안내**
- 앞에서 수정한 `df_work` 대신 원본 데이터를 복사하여 사용합니다.
- LSTM 실습은 아래 초기화 셀부터 순서대로 실행합니다.
- LSTM 실습을 처음부터 다시 진행할 때도 아래 초기화 셀부터 실행합니다.

In [ ]:
# ========================================
# LSTM 실습 데이터 초기화
# ========================================

train_df = df_raw.copy()
test_df = test_raw.copy()

print("LSTM 실습 데이터 준비 완료")
print("학습 데이터:", train_df.shape)
print("테스트 데이터:", test_df.shape)

-----

## **4-1 데이터를 학습에 적합하게 가공**

모델 학습에 적합한 형태로 데이터를 가공하는 과정은 예측 성능에 매우 중요합니다.




### **잔여 수명(RUL) 계산**

* 학습 데이터(`train_df`)는 각 엔진이 고장 날 때까지의 기록을 가지고 있습니다.
* 이를 이용해 각 시점에서의 RUL을 계산하여 타겟 변수로 사용합니다.
* RUL은 간단히 **'해당 엔진의 최대 사이클 - 현재 사이클'** 로 계산할 수 있습니다.

In [ ]:
# 각 엔진(unit_number)의 마지막 운전 주기(max_cycles) 구하기
# groupby(): 엔진별로 데이터를 그룹화
# max(): 각 엔진의 cycles 최댓값 계산
# reset_index(): 그룹별 계산 결과를 DataFrame으로 변환
max_cycles = train_df.groupby('unit_number')['cycles'].max().reset_index()


# 계산한 최댓값 컬럼의 이름 변경
max_cycles.columns = ['unit_number', 'max_cycles']


# 각 엔진의 최대 운전 주기를 원본 데이터에 추가
# unit_number가 같은 행끼리 연결하여 max_cycles 컬럼 추가
train_df = pd.merge(train_df, max_cycles, on='unit_number', how='left')


# 잔여수명(RUL) = 엔진의 마지막 운전 주기 - 현재 운전 주기
# 예: 1번 엔진의 마지막 주기가 192일 때
#     현재 주기가 150이면 RUL = 192 - 150 = 42
train_df['RUL'] = train_df['max_cycles'] - train_df['cycles']


# RUL 계산이 끝났으므로 max_cycles 컬럼 제거
train_df.drop(columns=['max_cycles'], inplace=True)

### **피처 선택 및 정규화**

* 모든 센서 데이터가 RUL 예측에 유용한 것은 아닙니다.
* 시간에 따라 변화가 거의 없는 센서는 모델 성능에 기여하지 못할 수 있습니다.
* 이러한 센서는 제거하는 것이 좋습니다. (예: 표준편차가 0에 가까운 센서)

* 그 후, 각 피처의 스케일이 다르므로 **정규화(Normalization)** 를 통해 데이터의 범위를 일치시켜 줍니다.
* `MinMaxScaler`를 사용하는 것이 일반적입니다.

In [ ]:
# 센서 및 운전 설정값의 표준편차 확인
sensor_cols = [
    col for col in train_df.columns
    if col.startswith(('sensor_', 'op_setting_'))
]

print(train_df[sensor_cols].std().sort_values())

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# 변화가 적은 센서 및 설정값 제거 (예시)
drop_cols = ['op_setting_3'] + [f'sensor_{i}' for i in [1, 5, 6, 10, 16, 18, 19]]
train_df.drop(columns=drop_cols, inplace=True)
test_df.drop(columns=drop_cols, inplace=True)


# 정규화할 피처 선택
features = [col for col in train_df.columns if col not in ['unit_number', 'cycles', 'RUL']]

# 스케일러 학습 및 적용
scaler = MinMaxScaler()
train_df[features] = scaler.fit_transform(train_df[features])
test_df[features] = scaler.transform(test_df[features])

### **시퀀스 데이터 생성 : Sliding Window**

* RNN은 순차적인 데이터를 입력으로 받는다.
* 따라서 시계열 데이터를 일정 길이의 **시퀀스(sequence)** 또는 **윈도우(window)** 로 잘라내야 한다.
* 예를 들어, 시퀀스 길이를 50으로 설정하면, 과거 50 사이클의 센서 데이터를 보고 현재 시점의 RUL을 예측하는 방식이다.

In [ ]:
import numpy as np

def create_sequences(df, sequence_length=50):
    sequences = []  # 모델 입력 데이터(X)를 저장할 리스트
    labels = []     # 정답 데이터(y)를 저장할 리스트

    # 엔진별로 시계열 데이터를 분리하여 처리
    # 서로 다른 엔진의 데이터가 하나의 시퀀스로 섞이지 않도록 함
    for unit in df['unit_number'].unique():

        # 현재 엔진의 센서 및 운전 조건 데이터 추출
        # .values: DataFrame에서 NumPy 배열로 변환
        unit_data = df[df['unit_number'] == unit][features].values

        # 현재 엔진의 RUL 값 추출
        unit_labels = df[df['unit_number'] == unit]['RUL'].values

        # Sliding Window로 일정 길이의 시퀀스 생성
        # 예: 데이터 100개, 윈도우 길이 50이면 총 51개 생성
        for i in range(len(unit_data) - sequence_length + 1):

            # i번째부터 sequence_length개만큼 데이터를 잘라 X에 저장
            # 예: i=0 → 0~49번 데이터, i=1 → 1~50번 데이터
            sequences.append(unit_data[i:i+sequence_length])

            # 각 시퀀스의 마지막 시점에 해당하는 RUL을 정답으로 저장
            # 예: i=0 → 49번 데이터의 RUL
            labels.append(unit_labels[i+sequence_length-1])

    # 리스트를 NumPy 배열로 변환하여 반환
    return np.array(sequences), np.array(labels)


# 하나의 시퀀스를 구성하는 운전 주기 수
sequence_length = 50

# 학습 데이터를 LSTM 입력 형태로 변환
X_train, y_train = create_sequences(train_df, sequence_length)

-----

## **4-2 RNN 모델 구축 및 학습**

이제 전처리된 데이터를 이용하여 RNN 모델을 만듭니다. 시계열 데이터 분석에 뛰어난 성능을 보이는 **LSTM(Long Short-Term Memory)** 모델을 사용하겠습니다.



### **모델 설계**

TensorFlow/Keras를 사용하여 간단한 LSTM 모델을 구성합니다.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential([
    LSTM(units=50, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),
    LSTM(units=50),
    Dropout(0.2),
    Dense(1)
])

model.compile(optimizer='adam', loss='mean_squared_error')
model.summary()

### **모델 학습**

설계한 모델에 준비된 학습 데이터를 넣어 학습을 진행합니다.

In [ ]:
history = model.fit(X_train, y_train, epochs=50, batch_size=64, validation_split=0.2)

In [ ]:
# 학습 결과 시각화해보기
# 손실이 감소하는지, 과적합 징후는 없는지 확인할 수 있다.

plt.figure(figsize=(8, 4))

plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')

plt.title('LSTM Training History')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.grid(True)
plt.show()

-----

## **4-3 모델 평가 및 예측**

학습된 모델의 성능을 평가하고, 테스트 데이터에 대한 RUL 예측을 수행합니다.



### **테스트 데이터 준비**

* 테스트 데이터도 학습 데이터와 동일한 방식으로 시퀀스 형태로 만들어야 합니다.
* 단, 테스트 데이터의 RUL을 예측할 때는 각 엔진의 **가장 마지막 시퀀스**만을 사용합니다.

In [ ]:
# 테스트 데이터의 마지막 시퀀스만 추출
X_test_sequences = []
for unit in test_df['unit_number'].unique():
    unit_data = test_df[test_df['unit_number'] == unit][features].values
    # 데이터 길이가 시퀀스 길이보다 짧을 경우 패딩 처리 가능
    if len(unit_data) < sequence_length:
        padded_data = np.zeros((sequence_length, unit_data.shape[1]))
        padded_data[-len(unit_data):] = unit_data
        X_test_sequences.append(padded_data)
    else:
        X_test_sequences.append(unit_data[-sequence_length:])

X_test = np.array(X_test_sequences)

### **예측 및 성능 평가**

* `RUL_FD001.txt` 파일에 있는 실제 RUL 값과 모델의 예측값을 비교하여 성능을 평가합니다.
* 주로 **RMSE(Root Mean Square Error)** 가 평가 지표로 사용됩니다.

In [ ]:
from sklearn.metrics import mean_squared_error

# RUL 예측
y_pred = model.predict(X_test)

# 실제 RUL 값 로드
y_true = pd.read_csv('/content/drive/MyDrive/_lec_data/_PdM/NASA/RUL_FD001.txt', header=None).values

# RMSE 계산
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
print(f"Test RMSE: {rmse}")

-----

## **4-4 다른 데이터셋 파일에 적용**

* 이 가이드를 바탕으로 데이터셋의 다른 파일(FD002, FD003, FD004)에 대해서도 동일한 과정을 적용해보고,
* LSTM 레이어의 수나 유닛 개수, 시퀀스 길이 등 하이퍼파라미터를 조정하며 성능을 개선해볼 수 있습니다.